# DA6401 Report - Section 2.2 Hyperparameter Sweep

This notebook runs the W&B random sweep for MNIST (target: `val/f1`) and launches the sweep agent.
Set `RUN_SWEEP=True` only when you intentionally want to start new runs.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from types import SimpleNamespace

import wandb

import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from train import train_and_evaluate
from utils.cli import validate_hidden_sizes

print('Project root:', ROOT)


In [ ]:
RUN_SWEEP = False  # Set True only when you intentionally want to launch sweep runs.

PROJECT = 'da6401_assignment_1'
ENTITY = 'da25s007-'
SWEEP_COUNT = 100

# Baseline/default args merged with per-run sweep config.
BASE_ARGS = {
    'dataset': 'mnist',
    'epochs': 10,
    'batch_size': 64,
    'loss': 'cross_entropy',
    'optimizer': 'rmsprop',
    'learning_rate': 0.001,
    'weight_decay': 0.0,
    'num_layers': 3,
    'hidden_size': [128, 128, 128],
    'activation': 'relu',
    'weight_init': 'xavier',
    'wandb_project': PROJECT,
    'wandb_entity': ENTITY,
    'wandb_mode': 'online',
    'seed': 42,
    'model_path': 'src/best_model.npy',
    'config_path': 'src/best_config.json',
}


def build_sweep_config(base_args):
    return {
        'name': 'da6401_mlp_sweep',
        'method': 'random',
        'metric': {'name': 'val/f1', 'goal': 'maximize'},
        'parameters': {
            'optimizer': {'values': ['sgd', 'momentum', 'nag', 'rmsprop']},
            'activation': {'values': ['relu', 'sigmoid', 'tanh']},
            'num_layers': {'values': [2, 3, 4, 5, 6]},
            'hidden_size': {'values': [[64], [96], [128], [64, 64, 64], [128, 128, 128]]},
            'learning_rate': {'values': [0.0005, 0.001, 0.003, 0.01]},
            'batch_size': {'values': [32, 64, 128]},
            'weight_decay': {'values': [0.0, 1e-5, 1e-4]},
            'weight_init': {'values': ['random', 'xavier', 'zeros']},
            'loss': {'values': ['cross_entropy', 'mean_squared_error']},
            'dataset': {'values': [base_args['dataset']]},
            'epochs': {'values': [8, 10, 12]},
        },
    }


def normalize_hidden_size(hidden_size, num_layers):
    if isinstance(hidden_size, int):
        return [hidden_size] * num_layers

    if isinstance(hidden_size, tuple):
        hidden_size = list(hidden_size)

    if not isinstance(hidden_size, list):
        return [128] * num_layers

    if len(hidden_size) == 0:
        return [128] * num_layers

    if len(hidden_size) == 1:
        return [int(hidden_size[0])] * num_layers

    if len(hidden_size) == num_layers:
        return [int(v) for v in hidden_size]

    return [int(hidden_size[0])] * num_layers


In [ ]:
if RUN_SWEEP:
    sweep_config = build_sweep_config(BASE_ARGS)
    sweep_id = wandb.sweep(sweep=sweep_config, project=PROJECT, entity=ENTITY)

    def train_fn():
        run = wandb.init()
        run_cfg = dict(run.config)

        args_dict = BASE_ARGS.copy()
        args_dict.update(run_cfg)
        merged_args = SimpleNamespace(**args_dict)

        merged_args.hidden_size = normalize_hidden_size(
            merged_args.hidden_size,
            int(merged_args.num_layers),
        )
        validate_hidden_sizes(merged_args)
        merged_args.wandb_mode = 'online'

        train_and_evaluate(merged_args, wandb_run_override=run)
        run.finish()

    print('Launching sweep:', sweep_id)
    wandb.agent(sweep_id, function=train_fn, count=SWEEP_COUNT)

    meta = {
        'project': PROJECT,
        'entity': ENTITY,
        'sweep_id': sweep_id,
        'count': SWEEP_COUNT,
    }
    out_path = ROOT / 'src' / 'tmp' / 'report_2_2_sweep_launch.json'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(meta, indent=2), encoding='utf-8')
    print('Wrote', out_path)
else:
    print('RUN_SWEEP=False -> no W&B sweep launched.')


In [ ]:
launch_path = ROOT / 'src' / 'tmp' / 'report_2_2_sweep_launch.json'
if launch_path.exists():
    info = json.loads(launch_path.read_text())
    print('Last sweep launch metadata:')
    print(json.dumps(info, indent=2))
else:
    print('No sweep launch metadata file found yet.')
